In [3]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score
import joblib
from tensorflow.keras.preprocessing import image

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    shear_range=0.2,
    zoom_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=False,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
img_size = (64, 64)
batch_size = 32
seed = 42

train_generator = train_datagen.flow_from_directory(
    'split_dataset/train',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=seed
)

val_generator = train_datagen.flow_from_directory(
    'split_dataset/train',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=True,
    seed=seed
)

test_generator = test_datagen.flow_from_directory(
    'split_dataset/test',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
print("\nClass names:", class_names)


Found 24794 images belonging to 36 classes.
Found 6184 images belonging to 36 classes.
Found 6640 images belonging to 36 classes.

Class names: ['क', 'क्ष', 'ख', 'ग', 'घ', 'ङ', 'च', 'छ', 'ज', 'ज्ञ', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'त्र्', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह']


In [6]:
def extract_features(generator):
    """Extract and flatten images with their labels"""
    features = []
    labels = []
    total_samples = generator.samples
    generator.reset()
    
    while len(features) * batch_size < total_samples:
        x, y = next(generator)
        features.append(x.reshape(x.shape[0], -1)) 
        labels.append(np.argmax(y, axis=1))  
    
    return np.vstack(features), np.concatenate(labels)

print("\nExtracting features...")
x_train, y_train = extract_features(train_generator)
x_val, y_val = extract_features(val_generator)
print("\nApplying PCA...")
pca = PCA(n_components=0.95)  
x_train_pca = pca.fit_transform(x_train)
x_val_pca = pca.transform(x_val)

print(f"Original dimension: {x_train.shape[1]}")
print(f"Reduced dimension: {x_train_pca.shape[1]}")



Extracting features...

Applying PCA...
Original dimension: 12288
Reduced dimension: 148


In [7]:
print("\nTraining KNN classifier...")
knn = KNeighborsClassifier(
    n_neighbors=5,  # Slightly higher for better generalization
    weights='distance',
    algorithm='kd_tree'
)
knn.fit(x_train_pca, y_train)


Training KNN classifier...


,n_neighbors,5
,weights,'distance'
,algorithm,'kd_tree'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [8]:
print("\nEvaluating model...")
y_pred = knn.predict(x_val_pca)

print("Validation Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))



Evaluating model...
Validation Accuracy: 0.8651358344113842

Classification Report:
              precision    recall  f1-score   support

           क       0.67      0.68      0.68       208
         क्ष       0.98      0.90      0.94       133
           ख       0.76      0.57      0.65       208
           ग       0.58      0.72      0.64       208
           घ       0.95      0.93      0.94       138
           ङ       0.91      0.73      0.81       131
           च       0.90      0.86      0.88       129
           छ       0.76      0.95      0.84       130
           ज       0.68      0.85      0.76       126
         ज्ञ       0.98      0.99      0.99       127
           झ       0.85      0.69      0.76       131
           ञ       0.51      0.96      0.66       140
           ट       0.89      0.89      0.89       204
           ठ       0.89      0.90      0.89       206
           ड       0.98      0.93      0.96       207
           ढ       0.99      0.98      0.99       

In [9]:
print("\nSaving models...")
joblib.dump(knn, 'knn_model.pkl')
joblib.dump(pca, 'pca_model.pkl')
print("Saved: knn_model.pkl and pca_model.pkl")


Saving models...
Saved: knn_model.pkl and pca_model.pkl


In [10]:
def predict_image(img_path):
    """Predict class for a single image"""
    # Load and preprocess image
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Load models
    knn = joblib.load('knn_model.pkl')
    pca = joblib.load('pca_model.pkl')
    
    # Transform and predict
    flat_img = img_array.reshape(1, -1)
    pca_img = pca.transform(flat_img)
    pred = knn.predict(pca_img)
    
    return class_names[pred[0]]